In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyomo.environ import ConcreteModel, Set, Param, Var, Objective, Constraint, SolverFactory, NonNegativeReals, minimize, Reals, Binary, value



## Get Data

In [11]:
#get fixed data
def get_fixed_data():
    """
    Returns the fixed data for the energy hub simulation.
    """
    num_timeslots = 24
    return {
        # Conversion efficiencies
        'conversion_p2h': 0.9,
        'conversion_h2p': 0.8,

        # Hydrogen storage capacity
        'hydrogen_capacity': 15,
        
        'p2h_max_rate': 5,
        'h2p_max_rate': 5,

        # Electrolyzer cost
        'electrolyzer_cost': 1, 

        # Wind model parameters
        'target_mean_wind': 4.5,
        'wind_reversion_strength': 0.15,
        'extreme_event_prob_wind': 0.03,

        # Price model parameters
        'mean_price': 35,
        'price_reversion_strength': 0.12,
        'wind_influence_on_price': -0.6,
        'price_cap': 90,  
        'price_floor': 0,
        
        
        'num_timeslots': num_timeslots,
        'demand_schedule': [5 + 2 * np.sin(2 * np.pi * t / 24) for t in range(num_timeslots)]

        
    }

data = get_fixed_data()

def price_model(current_price, previous_price, projected_wind, data):
    """
    Price process with dependence on previous prices and projected wind generation.

    Args:
        current_price (float): Current electricity price.
        previous_price (float): Electricity price at the previous time step.
        projected_wind (float): Projected wind generation for the next time step.
        data (dict): Fixed data containing model parameters.

    Returns:
        float: Next price.
    """
    mean_price = data['mean_price']
    reversion_strength = data['price_reversion_strength']
    wind_influence = data['wind_influence_on_price']
    price_cap = data['price_cap']
    price_floor = data['price_floor']

    mean_reversion = reversion_strength * (mean_price - current_price)
    wind_effect = wind_influence * projected_wind
    noise = np.random.normal(0, 1)

    next_price = current_price + 0.6 * (current_price - previous_price) + mean_reversion + wind_effect + noise

    if next_price < 0:
        if np.random.rand() > 0.2:
            next_price = np.random.uniform(0, mean_price * 0.3)

    return max(min(next_price, price_cap), price_floor)

def wind_model(current_wind, data):
    """
    Wind generation process with mean reversion and extreme events.

    Args:
        current_wind (float): Current wind generation.
        data (dict): Fixed data containing model parameters.

    Returns:
        float: Next wind generation.
    """
    target_mean = data['target_mean_wind']
    reversion_strength = data['wind_reversion_strength']
    extreme_event_prob = data['extreme_event_prob_wind']

    mean_reversion = reversion_strength * (target_mean - current_wind)
    noise = np.random.normal(0, 1)

    next_wind = current_wind + mean_reversion + noise

    if np.random.rand() < extreme_event_prob:
        next_wind += np.random.uniform(10, 20)

    return max(next_wind, 0)

def simulate_energy_hub(data, num_steps=100):
    """
    Simulates the energy hub over a specified number of time steps.

    Args:
        data (dict): Fixed data containing model parameters.
        num_steps (int): Number of time steps to simulate.

    Returns:
        pd.DataFrame: DataFrame containing the simulation results.
    """
    results = {
        'time': [],
        'price': [],
        'wind_generation': [],
        'demand': []
    }

    current_price = data['mean_price']
    previous_price = current_price
    current_wind = data['target_mean_wind']

    for t in range(num_steps):
        projected_wind = wind_model(current_wind, data)
        next_price = price_model(current_price, previous_price, projected_wind, data)

        results['time'].append(t)
        results['price'].append(next_price)
        results['wind_generation'].append(projected_wind)
        results['demand'].append(data['demand_schedule'][t % data['num_timeslots']])

        previous_price = current_price
        current_price = next_price
        current_wind = projected_wind

    return pd.DataFrame(results)

# Run the simulation
#simulation_results = simulate_energy_hub(data, num_steps=100)
#print(simulation_results.head())


## Pseudo code

In [24]:
class EnergySystemValidator:
    def __init__(self, p2h_max, h2p_max, r_h2p_efficiency):
        self.p2h_max = p2h_max
        self.h2p_max = h2p_max
        self.r_h2p = r_h2p_efficiency

    def check_feasibility(self, state, decision, demand):
        """
        Returns (is_valid, error_message)
        """
        # 1. Capacity Checks
        if decision['p_p2h'] > self.p2h_max:
            return False, "Exceeds P2H capacity"
        
        if decision['h_h2p'] > self.h2p_max:
            return False, "Exceeds H2P capacity"

        # 2. Grid Balancing Check
        # The grid must balance the wind, demand, and hydrogen conversions
        expected_grid = (demand + decision['p_p2h'] - 
                         state['p_wind'] - (self.r_h2p * decision['h_h2p']))
        
        if abs(decision['p_grid'] - expected_grid) > 1e-6:
            return False, f"Power imbalance: Grid={decision['p_grid']}, Expected={expected_grid}"

        # 3. Binary and Switching Logic
        y_act, y_de = decision['y_act'], decision['y_de']
        
        if y_act not in [0, 1] or y_de not in [0, 1]:
            return False, "Activation signals must be 0 or 1"
            
        if (y_act * y_de) != 0:
            return False, "Cannot activate and deactivate at the same time"

        return True, "Valid"
    
    @staticmethod
    def get_dummy_action(state, demand, params):

        
    #"""
    #Creates a safe, 'do-nothing' feasible action when the policy fails.
    # This ensures the simulation can continue without crashing.
    #"""
    # 1. Default to no hydrogen conversion
        p_p2h_dummy = 0.0
        h_h2p_dummy = 0.0
    
    # 2. Binary signals: Stay in current state (no activation or deactivation)
        y_act_dummy = 0
        y_de_dummy = 0
    
    # 3. Force Grid Balance
    # Following the equation: p_grid = D_t + p_p2h - p_wind - (R_h2p * h_h2p)
    # With dummy hydrogen values at 0, the grid simply covers net demand
        p_grid_dummy = demand - state['p_wind']
    
        return {
        'p_p2h': p_p2h_dummy,
        'h_h2p': h_h2p_dummy,
        'p_grid': p_grid_dummy,
        'y_act': y_act_dummy,
        'y_de': y_de_dummy
    }

    def decision_to_dict(self, decision):
        return {
            'p_p2h': decision['p_p2h'],
            'h_h2p': decision['h_h2p'],
            'p_grid': decision['p_grid'],
            'y_act': decision['y_act'],
            'y_de': decision['y_de']
        }
    
    @staticmethod
    def cost_function(decision, price):
        """
        Calculates the cost of a given decision based on the current price and parameters.
        """
        # Cost from grid consumption
        grid_cost = decision['p_grid'] * price

        
        return grid_cost
    
    @staticmethod
    def apply_dynamics(state, u, data):
        """
        Applies the transition dynamics to move from state t to state t+1.
        
        Args:
            state (dict): Current state variables {h, y, p_wind, lambda_grid, ...}
            u (dict): Decisions made at time t {p_p2h, h_h2p, y_act, y_de}
            data (dict): Parameters including efficiencies and model constants
            
        Returns:
            dict: The updated state for the next time step (t+1)
        """
        next_state = {}
        
        # 1. Hydrogen Storage Transition: h_{t+1} = h_t + (p_p2h * R_p2h) - h_h2p
        # R_p2h is the 'conversion_p2h' (0.9) from your fixed data
        next_state['h'] = state['h'] + (u['p_p2h'] * data['conversion_p2h']) - u['h_h2p']
        
        # 2. System Status Transition: y_{t+1} = y_t + y_act - y_de
        next_state['y'] = state['y'] + u['y_act'] - u['y_de']
        
        # 3. Store current values as 'previous' for the next step's stochastic models
        next_state['p_wind_prev'] = state['p_wind']
        next_state['lambda_grid_prev'] = state['lambda_grid']
        
        # 4. Stochastic Wind Transition: p_{t+1} ~ P(.|p_t)
        # Uses your wind_model with mean reversion and extreme event logic
        next_state['p_wind'] = wind_model(state['p_wind'], data)
        
        # 5. Stochastic Price Transition: lambda_{t+1} ~ P(.|lambda_t, lambda_prev, wind_next)
        # Uses your price_model which depends on the newly sampled wind
        next_state['lambda_grid'] = price_model(
            current_price=state['lambda_grid'],
            previous_price=state['lambda_grid_prev'],
            projected_wind=next_state['p_wind'],
            #previous_wind=next_state['p_wind_prev'],
            data=data
        )
        
        return next_state

In [33]:
for t in range(1, 24):
    init_state = {
        'h': 5.0,  # Initial hydrogen storage level
        'y': 0,    # Initial system status (inactive)
        'p_wind': 4.0,  # Initial wind generation
        'lambda_grid': 30.0,  # Initial electricity price
        'p_wind_prev': 4.0,  # Previous wind generation
        'lambda_grid_prev': 30.0  # Previous electricity price
    }

    dummy_decision = EnergySystemValidator.get_dummy_action(init_state, data['demand_schedule'][t], data)
    state = EnergySystemValidator.apply_dynamics(init_state, dummy_decision, data)
    decision = EnergySystemValidator.get_dummy_action(state, data['demand_schedule'][t], data)
    # skipping check feasiibility since dummy action is designed to be feasible
    cost = EnergySystemValidator.cost_function(decision, state['lambda_grid'])
    next_state = EnergySystemValidator.apply_dynamics(state, decision, data) 
    print(f"Time step {t}:")
    print(f"cost: {cost:.2f}, wind: {state['p_wind']:.2f}, price: {state['lambda_grid']:.2f}")
    print(f"demand: {data['demand_schedule'][t]}")
    



Time step 1:
cost: 40.41, wind: 4.02, price: 27.06
demand: 5.5176380902050415
Time step 2:
cost: 20.60, wind: 5.21, price: 26.12
demand: 6.0
Time step 3:
cost: 87.97, wind: 3.24, price: 27.75
demand: 6.414213562373095
Time step 4:
cost: 49.11, wind: 4.88, price: 26.50
demand: 6.732050807568877
Time step 5:
cost: 75.55, wind: 4.32, price: 28.95
demand: 6.931851652578136
Time step 6:
cost: 45.43, wind: 5.22, price: 25.49
demand: 7.0
Time step 7:
cost: 45.44, wind: 5.32, price: 28.19
demand: 6.931851652578136
Time step 8:
cost: 51.05, wind: 4.70, price: 25.10
demand: 6.732050807568878
Time step 9:
cost: 81.80, wind: 3.53, price: 28.33
demand: 6.414213562373095
Time step 10:
cost: -17.34, wind: 6.64, price: 27.03
demand: 6.0
Time step 11:
cost: 61.47, wind: 3.35, price: 28.40
demand: 5.517638090205042
Time step 12:
cost: -303.03, wind: 24.76, price: 15.33
demand: 5.0
Time step 13:
cost: 17.11, wind: 3.88, price: 28.58
demand: 4.4823619097949585
Time step 14:
cost: -16.88, wind: 4.64, price